# Pneumonia Detection Model — Colab Training Notebook

This notebook trains a transfer learning model for chest X-ray pneumonia classification using the Kaggle dataset:

`paultimothymooney/chest-xray-pneumonia`

It is designed for Google Colab and exports files that can be reused in a real project.

Final outputs:

- `pneumonia_model.keras`
- `pneumonia_model.tflite`
- `class_indices.json`
- `training_history.json`

> Important: this model is for educational/prototype use, not medical diagnosis.

## 1. Install and import dependencies

In [ ]:
!pip -q install kagglehub scikit-learn


In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print('TensorFlow:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)


## 2. Reproducibility and configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

INITIAL_EPOCHS = 8
FINE_TUNE_EPOCHS = 6

MODEL_NAME = 'pneumonia_model'


## 3. Download Kaggle dataset

In [ ]:
DATASET_ROOT = Path(kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia'))
print('Dataset root:', DATASET_ROOT)

possible_roots = [
    DATASET_ROOT / 'chest_xray',
    DATASET_ROOT,
    Path('/content/chest_xray'),
]

DATA_DIR = next((p for p in possible_roots if (p / 'train').exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError('Could not find chest_xray/train directory')

TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR = DATA_DIR / 'val'
TEST_DIR = DATA_DIR / 'test'

print('Train:', TRAIN_DIR)
print('Val:', VAL_DIR)
print('Test:', TEST_DIR)

for folder in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    print('\n', folder)
    for cls in sorted(os.listdir(folder)):
        cls_path = folder / cls
        if cls_path.is_dir():
            print(cls, len(list(cls_path.glob('*'))))


## 4. Data generators

We use `preprocess_input` from DenseNet instead of simple `/255` because the model was pretrained with this preprocessing.

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.10,
    shear_range=0.05,
    horizontal_flip=False,
    fill_mode='nearest'
)

eval_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED
)

val_generator = eval_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_generator = eval_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

class_indices = train_generator.class_indices
print('Class indices:', class_indices)

with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=4)


## 5. Handle class imbalance

In [ ]:
classes = np.unique(train_generator.classes)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_generator.classes
)

class_weights = {int(cls): float(weight) for cls, weight in zip(classes, weights)}
print('Class weights:', class_weights)


## 6. Build transfer learning model

In [ ]:
base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

model.summary()


## 7. Train classifier head

In [ ]:
callbacks = [
    ModelCheckpoint(
        f'{MODEL_NAME}_best.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history_1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=INITIAL_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)


## 8. Fine-tune last layers

In [ ]:
base_model.trainable = True

# Keep most layers frozen and fine-tune only the last part.
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

history_2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)


## 9. Plot training curves

In [ ]:
def merge_histories(*histories):
    merged = {}
    for hist in histories:
        for key, values in hist.history.items():
            merged.setdefault(key, []).extend(values)
    return merged

history = merge_histories(history_1, history_2)

with open('training_history.json', 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in history.items()}, f, indent=4)

def plot_metric(metric):
    plt.figure(figsize=(8, 5))
    plt.plot(history[metric], label=f'train {metric}')
    val_metric = f'val_{metric}'
    if val_metric in history:
        plt.plot(history[val_metric], label=f'validation {metric}')
    plt.xlabel('Epoch')
    plt.ylabel(metric)
    plt.title(metric)
    plt.legend()
    plt.grid(True)
    plt.show()

plot_metric('accuracy')
plot_metric('loss')
plot_metric('auc')


## 10. Evaluate on test set

In [ ]:
test_results = model.evaluate(test_generator, verbose=1)
print(dict(zip(model.metrics_names, test_results)))

y_true = test_generator.classes
y_prob = model.predict(test_generator).ravel()
y_pred = (y_prob >= 0.5).astype(int)

idx_to_class = {v: k for k, v in class_indices.items()}
target_names = [idx_to_class[i] for i in sorted(idx_to_class)]

print('ROC AUC:', roc_auc_score(y_true, y_prob))
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=target_names))

cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix:')
print(cm)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title('Confusion Matrix')
plt.xticks([0, 1], target_names)
plt.yticks([0, 1], target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.colorbar()
plt.show()


## 11. Save model for real project use

In [ ]:
model.save(f'{MODEL_NAME}.keras')
print('Saved:', f'{MODEL_NAME}.keras')

# Optional: save a TensorFlow SavedModel export
model.export(f'{MODEL_NAME}_savedmodel')
print('Exported SavedModel folder:', f'{MODEL_NAME}_savedmodel')


## 12. Convert to TensorFlow Lite

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open(f'{MODEL_NAME}.tflite', 'wb') as f:
    f.write(tflite_model)

print('Saved:', f'{MODEL_NAME}.tflite')
print('TFLite size MB:', len(tflite_model) / (1024 * 1024))


## 13. Test single image prediction

In [ ]:
from tensorflow.keras.preprocessing import image

def predict_image(image_path, threshold=0.5):
    img = image.load_img(image_path, target_size=IMG_SIZE)
    arr = image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr)

    prob = float(model.predict(arr, verbose=0)[0][0])
    pred_idx = int(prob >= threshold)
    label = idx_to_class[pred_idx]

    return {
        'label': label,
        'pneumonia_probability': prob,
        'threshold': threshold
    }

# Example:
# sample_path = next((TEST_DIR / 'PNEUMONIA').glob('*'))
# predict_image(sample_path)


## 14. Download final files from Colab

In [ ]:
from google.colab import files

for file_name in [
    f'{MODEL_NAME}.keras',
    f'{MODEL_NAME}.tflite',
    f'{MODEL_NAME}_best.keras',
    'class_indices.json',
    'training_history.json'
]:
    if Path(file_name).exists():
        print('Downloading:', file_name)
        files.download(file_name)
    else:
        print('Missing:', file_name)
